## 1. Setup and reproducibility

Import the libraries used for loading data, preparing features, fitting classifiers and assessing their performance. The fitted preprocessing pipelines keep imputation and scaling within each training fold.

In [1]:
# ============================================================
# SPACESHIP TITANIC
# Baseline Classification Model: Logistic Regression
# ============================================================

# Import pandas for data manipulation.
import pandas as pd

# Import numpy for numerical operations.
import numpy as np

# Import Path for managing file locations.
from pathlib import Path

# Import train_test_split to create training and validation sets.
from sklearn.model_selection import train_test_split

# Import tools for preparing numerical and categorical features.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Import methods for handling missing values and scaling.
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Import Logistic Regression.
from sklearn.linear_model import LogisticRegression

# Import classification evaluation metrics.
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Load the competition data

Load the labelled **train.csv** and unlabelled **test.csv** from the project directory. The notebook supports execution from either the project root or its `notebooks/` subdirectory.

In [2]:
# Locate the competition CSV files in the project root or its parent.
# This supports notebooks stored either at project root or in notebooks/.
working_dir = Path.cwd()
project_root = next(
    (folder for folder in (working_dir, working_dir.parent)
     if (folder / "train.csv").is_file()
     and (folder / "test.csv").is_file()),
    None
)
if project_root is None:
    raise FileNotFoundError(
        "Could not find train.csv and test.csv in the current folder "
        "or its parent. Download the Spaceship Titanic files and "
        "place them in the project's root directory."
    )

# Load the competition data.
train_df = pd.read_csv(project_root / "train.csv")
test_df = pd.read_csv(project_root / "test.csv")

# Verify that both datasets loaded correctly.
print("Training dataset:", train_df.shape)
print("Competition test dataset:", test_df.shape)

Training dataset: (8693, 14)
Competition test dataset: (4277, 13)


## 3. Extract cabin features

Split the original `Cabin` text into **CabinDeck**, **CabinNumber**, and **CabinSide** so the models can use spatial information without treating the original text as an undifferentiated identifier.

In [3]:
# ============================================================
# STEP 3: EXTRACT CABIN FEATURES
# ============================================================

# Define a reusable function to extract information
# from the original Cabin column.

def extract_cabin_features(df):

    # Create a copy to avoid changing the original dataset.
    df = df.copy()

    # Split Cabin into three components using "/".
    # Example: "B/0/P" becomes ["B", "0", "P"].
    cabin_parts = df["Cabin"].str.split("/", expand=True)

    # Extract the cabin deck.
    df["CabinDeck"] = cabin_parts[0]

    # Extract the cabin number and convert it to numeric.
    # Invalid or missing values become NaN.
    df["CabinNumber"] = pd.to_numeric(
        cabin_parts[1],
        errors="coerce"
    )

    # Extract the cabin side.
    df["CabinSide"] = cabin_parts[2]

    # Remove the original Cabin column because
    # its information is now represented by three features.
    df = df.drop(columns=["Cabin"])

    # Return the updated DataFrame.
    return df


# Apply the same feature extraction to both datasets.
train_model_df = extract_cabin_features(train_df)

test_model_df = extract_cabin_features(test_df)


# Display the new dataset dimensions.
print("Training dataset:", train_model_df.shape)
print("Test dataset:", test_model_df.shape)

# Display the extracted cabin features.
display(
    train_model_df[
        ["CabinDeck", "CabinNumber", "CabinSide"]
    ].head(10)
)

Training dataset: (8693, 16)
Test dataset: (4277, 15)


,CabinDeck,CabinNumber,CabinSide
0,B,0.0,P
1,F,0.0,S
2,A,0.0,S
3,A,0.0,S
4,F,1.0,S
5,F,0.0,P
6,F,2.0,S
7,G,0.0,S
8,F,3.0,S
9,B,1.0,P


## 4. Define predictors and target

Separate `Transported` from the 13 modelling predictors. Passenger identifiers and the passenger name are excluded from this baseline feature set.

In [4]:
# ============================================================
# STEP 4: SEPARATE PREDICTORS (X) AND TARGET (y)
# ============================================================

# Define the numerical predictor variables.
numerical_features = [
    "Age",
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck",
    "CabinNumber"
]

# Define the categorical predictor variables.
categorical_features = [
    "HomePlanet",
    "CryoSleep",
    "Destination",
    "VIP",
    "CabinDeck",
    "CabinSide"
]

# Combine numerical and categorical features.
selected_features = numerical_features + categorical_features

# X contains the passenger characteristics used for prediction.
X = train_model_df[selected_features].copy()

# y contains the correct transportation outcomes.
y = train_model_df["Transported"].copy()

# Prepare the competition test predictors using the same features.
X_test = test_model_df[selected_features].copy()

# Display the dimensions of our predictors and target.
print("Training predictors (X):", X.shape)
print("Training target (y):", y.shape)
print("Competition test predictors:", X_test.shape)

# Confirm that the target is not included among the predictors.
print("\nTarget included in X:", "Transported" in X.columns)

# Display the first five observations.
display(X.head())

Training predictors (X): (8693, 13)
Training target (y): (8693,)
Competition test predictors: (4277, 13)

Target included in X: False


,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,CabinNumber,HomePlanet,CryoSleep,Destination,VIP,CabinDeck,CabinSide
0,39.0,0.0,0.0,0.0,0.0,0.0,0.0,Europa,False,TRAPPIST-1e,False,B,P
1,24.0,109.0,9.0,25.0,549.0,44.0,0.0,Earth,False,TRAPPIST-1e,False,F,S
2,58.0,43.0,3576.0,0.0,6715.0,49.0,0.0,Europa,False,TRAPPIST-1e,True,A,S
3,33.0,0.0,1283.0,371.0,3329.0,193.0,0.0,Europa,False,TRAPPIST-1e,False,A,S
4,16.0,303.0,70.0,151.0,565.0,2.0,1.0,Earth,False,TRAPPIST-1e,False,F,S


## 5. Create a stratified development split

Reserve a stratified 20% validation split and use the remaining 80% as the development-training subset. Later model selection also uses five-fold cross-validation *within* that training subset.

In [5]:
# ============================================================
# STEP 5: SPLIT DATA INTO TRAINING AND VALIDATION SETS
# ============================================================

# Divide the labelled dataset into training and validation subsets.
X_train, X_valid, y_train, y_valid = train_test_split(
    X,                  # Predictor variables
    y,                  # Target variable
    test_size=0.20,     # Reserve 20% for validation
    random_state=42,    # Ensure reproducible results
    stratify=y          # Preserve target-class proportions
)

# Display the dimensions of each dataset.
print("TRAINING AND VALIDATION DATASET DIMENSIONS")
print("-" * 50)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)

print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)

TRAINING AND VALIDATION DATASET DIMENSIONS
--------------------------------------------------
X_train: (6954, 13)
X_valid: (1739, 13)
y_train: (6954,)
y_valid: (1739,)


In [6]:
# ============================================================
# CHECK CLASS DISTRIBUTION AFTER STRATIFIED SPLITTING
# ============================================================

# Calculate target percentages in the original dataset.
original_distribution = (
    y.value_counts(normalize=True) * 100
)

# Calculate target percentages in the training subset.
training_distribution = (
    y_train.value_counts(normalize=True) * 100
)

# Calculate target percentages in the validation subset.
validation_distribution = (
    y_valid.value_counts(normalize=True) * 100
)

# Combine the distributions into one table.
distribution_comparison = pd.DataFrame({
    "Original (%)": original_distribution,
    "Training (%)": training_distribution,
    "Validation (%)": validation_distribution
})

# Display percentages rounded to two decimal places.
display(distribution_comparison.round(2))

,Original (%),Training (%),Validation (%)
Transported,,,
True,50.36,50.36,50.37
False,49.64,49.64,49.63


## 6. Original preprocessing: imputation, scaling, and encoding

Numerical missing values are replaced using **training-fold medians** before standardization. Categorical missing values are replaced with training-fold modes and then one-hot encoded. These steps occur inside each classifier pipeline to reduce preprocessing leakage.

In [7]:
# ============================================================
# STEP 6A: NUMERICAL PREPROCESSING PIPELINE
# ============================================================

# Create a pipeline for numerical predictors.
numerical_pipeline = Pipeline(
    steps=[
        # Replace missing numerical values with the median
        # calculated from the training subset.
        ("imputer", SimpleImputer(strategy="median")),

        # Standardize numerical variables so they have
        # comparable scales during model training.
        ("scaler", StandardScaler())
    ]
)

print("Numerical preprocessing pipeline created!")

Numerical preprocessing pipeline created!


In [8]:
# ============================================================
# STEP 6B: CATEGORICAL PREPROCESSING PIPELINE
# ============================================================

# Create a pipeline for categorical predictors.
categorical_pipeline = Pipeline(
    steps=[
        # Replace missing categorical values with the
        # most frequently occurring category in training.
        ("imputer", SimpleImputer(strategy="most_frequent")),

        # Convert categorical values into numerical indicators.
        # handle_unknown="ignore" prevents errors when validation
        # or test data contain previously unseen categories.
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

print("Categorical preprocessing pipeline created!")

Categorical preprocessing pipeline created!


In [9]:
# ============================================================
# STEP 6C: COMBINE NUMERICAL AND CATEGORICAL PREPROCESSING
# ============================================================

# ColumnTransformer applies different preprocessing
# operations to different groups of columns.
preprocessor = ColumnTransformer(
    transformers=[
        # Apply numerical preprocessing to numerical columns.
        ("num", numerical_pipeline, numerical_features),

        # Apply categorical preprocessing to categorical columns.
        ("cat", categorical_pipeline, categorical_features)
    ]
)

print("Combined preprocessing pipeline created!")

Combined preprocessing pipeline created!


## 7. Logistic Regression baseline

Convert the original boolean categories to strings for consistent categorical processing, train Logistic Regression, and evaluate its initial validation predictions.

In [10]:
# ============================================================
# STEP 7A: PREPARE BOOLEAN CATEGORICAL VARIABLES
# ============================================================

# Create copies so the original training and validation
# datasets remain unchanged.
X_train_model = X_train.copy()
X_valid_model = X_valid.copy()

# Identify categorical columns containing Boolean values.
boolean_columns = ["CryoSleep", "VIP"]

# Convert Boolean values to string categories.
# Missing values are preserved for the imputer.
for column in boolean_columns:

    X_train_model[column] = (
        X_train_model[column].astype("string").astype(object)
    )

    X_valid_model[column] = (
        X_valid_model[column].astype("string").astype(object)
    )

# Replace pandas missing-value markers with np.nan
# to ensure compatibility with SimpleImputer.
for column in boolean_columns:

    X_train_model[column] = (
        X_train_model[column].where(
            X_train_model[column].notna(),
            np.nan
        )
    )

    X_valid_model[column] = (
        X_valid_model[column].where(
            X_valid_model[column].notna(),
            np.nan
        )
    )

print("Boolean categorical variables prepared successfully!")

Boolean categorical variables prepared successfully!


In [11]:
# ============================================================
# STEP 7B: CREATE LOGISTIC REGRESSION PIPELINE
# ============================================================

# Combine preprocessing and Logistic Regression
# into one complete machine learning pipeline.
logistic_model = Pipeline(
    steps=[
        # Step 1: Apply numerical and categorical preprocessing.
        ("preprocessor", preprocessor),

        # Step 2: Train a Logistic Regression classifier.
        ("classifier", LogisticRegression(
            max_iter=2000,
            random_state=42
        ))
    ]
)

print("Logistic Regression pipeline created successfully!")

Logistic Regression pipeline created successfully!


In [12]:
# ============================================================
# STEP 7C: TRAIN THE LOGISTIC REGRESSION MODEL
# ============================================================

# Fit the complete pipeline using the training subset only.
# The preprocessing pipeline learns its parameters from X_train_model.
# Logistic Regression then learns relationships between
# the transformed predictors and transportation outcomes.
logistic_model.fit(X_train_model, y_train)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


In [13]:
# ============================================================
# STEP 7D: GENERATE VALIDATION PREDICTIONS
# ============================================================

# Predict whether each validation passenger was transported.
y_valid_pred = logistic_model.predict(X_valid_model)

# Display the first ten predictions.
print("First ten predictions:")

print(y_valid_pred[:10])

# Display the corresponding actual outcomes.
print("\nFirst ten actual outcomes:")

print(y_valid.iloc[:10].to_numpy())

First ten predictions:
[ True  True False  True False  True  True  True False  True]

First ten actual outcomes:
[ True False False  True False False  True  True False  True]


In [14]:
# ============================================================
# STEP 7E: EVALUATE BASELINE CLASSIFICATION ACCURACY
# ============================================================

# Calculate the proportion of correct predictions.
baseline_accuracy = accuracy_score(
    y_valid,
    y_valid_pred
)

# Display the accuracy as a percentage.
print(
    f"Logistic Regression Validation Accuracy: "
    f"{baseline_accuracy * 100:.2f}%"
)

Logistic Regression Validation Accuracy: 78.84%


## 8. Classification diagnostics

Inspect the confusion matrix and class-level precision, recall and F1-score; accuracy alone can conceal different kinds of classification errors.

In [15]:
# ============================================================
# STEP 8A: CALCULATE THE CONFUSION MATRIX
# ============================================================

# Calculate the confusion matrix using actual and predicted outcomes.
# labels=[False, True] ensures a consistent ordering of classes.
cm = confusion_matrix(
    y_valid,
    y_valid_pred,
    labels=[False, True]
)

# Convert the confusion matrix into a labelled DataFrame
# to make the results easier to interpret.
cm_df = pd.DataFrame(
    cm,
    index=["Actual False", "Actual True"],
    columns=["Predicted False", "Predicted True"]
)

# Display the confusion matrix.
display(cm_df)

,Predicted False,Predicted True
Actual False,669,194
Actual True,174,702


In [16]:
# ============================================================
# STEP 8B: GENERATE A CLASSIFICATION REPORT
# ============================================================

# Generate precision, recall, F1-score, and support
# for both transportation classes.
report = classification_report(
    y_valid,
    y_valid_pred,
    labels=[False, True],
    target_names=["Not Transported", "Transported"],
    digits=4
)

# Display the classification report.
print("LOGISTIC REGRESSION CLASSIFICATION REPORT")
print("-" * 50)

print(report)

LOGISTIC REGRESSION CLASSIFICATION REPORT
--------------------------------------------------
                 precision    recall  f1-score   support

Not Transported     0.7936    0.7752    0.7843       863
    Transported     0.7835    0.8014    0.7923       876

       accuracy                         0.7884      1739
      macro avg     0.7885    0.7883    0.7883      1739
   weighted avg     0.7885    0.7884    0.7883      1739



In [17]:
# ============================================================
# STEP 8C: CREATE OUR FIRST EXPERIMENT RECORD
# ============================================================

# Create a DataFrame to record baseline model performance.
experiment_log = pd.DataFrame([
    {
        "Experiment": "Baseline_01",
        "Model": "Logistic Regression",
        "Features": len(selected_features),
        "Validation_Method": "Stratified 80/20",
        "Validation_Accuracy": baseline_accuracy
    }
])

# Display the experiment results as percentages.
experiment_display = experiment_log.copy()

experiment_display["Validation_Accuracy"] = (
    experiment_display["Validation_Accuracy"] * 100
).round(2)

display(experiment_display)

,Experiment,Model,Features,Validation_Method,Validation_Accuracy
0,Baseline_01,Logistic Regression,13,Stratified 80/20,78.84


## 9. Random Forest comparison

Fit a tree-ensemble alternative and compare its training and validation results to the Logistic Regression baseline.

In [18]:
# ============================================================
# STEP 9A: IMPORT RANDOM FOREST CLASSIFIER
# ============================================================

# Import Random Forest for classification.
from sklearn.ensemble import RandomForestClassifier

print("Random Forest imported successfully!")

Random Forest imported successfully!


In [19]:
# ============================================================
# STEP 9B: CREATE RANDOM FOREST PIPELINE
# ============================================================

# Create a pipeline that combines preprocessing
# with a Random Forest classification model.
random_forest_model = Pipeline(
    steps=[
        # Apply the existing preprocessing steps.
        ("preprocessor", preprocessor),

        # Train a forest containing 200 decision trees.
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

print("Random Forest pipeline created successfully!")

Random Forest pipeline created successfully!


In [20]:
# ============================================================
# STEP 9C: TRAIN RANDOM FOREST CLASSIFIER
# ============================================================

# Train the Random Forest pipeline using the same
# training observations used for Logistic Regression.
random_forest_model.fit(
    X_train_model,
    y_train
)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [21]:
# ============================================================
# STEP 9D: RANDOM FOREST VALIDATION PREDICTIONS
# ============================================================

# Predict transportation outcomes for validation passengers.
rf_valid_pred = random_forest_model.predict(
    X_valid_model
)

# Display the first ten predictions.
print("First ten Random Forest predictions:")
print(rf_valid_pred[:10])

First ten Random Forest predictions:
[ True  True False  True False  True  True False False  True]


In [22]:
# ============================================================
# STEP 9E: EVALUATE RANDOM FOREST
# ============================================================

# Calculate validation accuracy.
rf_accuracy = accuracy_score(
    y_valid,
    rf_valid_pred
)

# Display validation accuracy.
print(
    f"Random Forest Validation Accuracy: "
    f"{rf_accuracy * 100:.2f}%"
)

# Display detailed classification metrics.
print("\nRANDOM FOREST CLASSIFICATION REPORT")
print("-" * 50)

print(
    classification_report(
        y_valid,
        rf_valid_pred,
        labels=[False, True],
        target_names=["Not Transported", "Transported"],
        digits=4
    )
)

Random Forest Validation Accuracy: 80.68%

RANDOM FOREST CLASSIFICATION REPORT
--------------------------------------------------
                 precision    recall  f1-score   support

Not Transported     0.7800    0.8505    0.8137       863
    Transported     0.8383    0.7637    0.7993       876

       accuracy                         0.8068      1739
      macro avg     0.8092    0.8071    0.8065      1739
   weighted avg     0.8094    0.8068    0.8065      1739



In [23]:
# ============================================================
# STEP 9F: COMPARE CLASSIFICATION MODELS
# ============================================================

# Create a table containing the validation accuracy
# of our two classification algorithms.
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],

    "Validation_Accuracy": [
        baseline_accuracy,
        rf_accuracy
    ]
})

# Convert accuracy values into percentages.
model_comparison["Validation_Accuracy"] = (
    model_comparison["Validation_Accuracy"] * 100
)

# Sort models from highest to lowest accuracy.
model_comparison = model_comparison.sort_values(
    by="Validation_Accuracy",
    ascending=False
)

# Display the comparison.
display(model_comparison.round(2))

,Model,Validation_Accuracy
1,Random Forest,80.68
0,Logistic Regression,78.84


## 10. Detect possible overfitting

Compare training accuracy with validation accuracy. A large gap, particularly for an unrestricted Random Forest, can indicate that a model fits training-specific patterns.

In [24]:
# ============================================================
# STEP 10A: EVALUATE RANDOM FOREST ON TRAINING DATA
# ============================================================

# Generate predictions for the observations used to train
# our Random Forest model.
rf_train_pred = random_forest_model.predict(X_train_model)

# Calculate training accuracy.
rf_train_accuracy = accuracy_score(
    y_train,
    rf_train_pred
)

# Display training and validation accuracy.
print(
    f"Random Forest Training Accuracy: "
    f"{rf_train_accuracy * 100:.2f}%"
)

print(
    f"Random Forest Validation Accuracy: "
    f"{rf_accuracy * 100:.2f}%"
)

Random Forest Training Accuracy: 99.94%
Random Forest Validation Accuracy: 80.68%


In [25]:
# ============================================================
# STEP 10B: CALCULATE GENERALIZATION GAP
# ============================================================

# Calculate the difference between training and validation
# accuracy in percentage points.
rf_generalization_gap = (
    rf_train_accuracy - rf_accuracy
) * 100

print(
    f"Random Forest Generalization Gap: "
    f"{rf_generalization_gap:.2f} percentage points"
)

Random Forest Generalization Gap: 19.26 percentage points


In [26]:
# ============================================================
# STEP 11: COMPARE TRAINING AND VALIDATION PERFORMANCE
# ============================================================

# Generate Logistic Regression predictions on training data.
log_train_pred = logistic_model.predict(X_train_model)

# Calculate Logistic Regression training accuracy.
log_train_accuracy = accuracy_score(
    y_train,
    log_train_pred
)

# Create a comparison table for both algorithms.
generalization_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Training_Accuracy": [
        log_train_accuracy,
        rf_train_accuracy
    ],
    "Validation_Accuracy": [
        baseline_accuracy,
        rf_accuracy
    ]
})

# Calculate the generalization gap.
generalization_comparison["Generalization_Gap"] = (
    generalization_comparison["Training_Accuracy"]
    - generalization_comparison["Validation_Accuracy"]
)

# Convert the metrics into percentages.
percentage_columns = [
    "Training_Accuracy",
    "Validation_Accuracy",
    "Generalization_Gap"
]

generalization_comparison[percentage_columns] = (
    generalization_comparison[percentage_columns] * 100
)

# Display the comparison rounded to two decimal places.
display(generalization_comparison.round(2))

,Model,Training_Accuracy,Validation_Accuracy,Generalization_Gap
0,Logistic Regression,79.03,78.84,0.20
1,Random Forest,99.94,80.68,19.26


## 11. Regularize Random Forest

Control model complexity and compare performance to the earlier Random Forest configuration. Feature selection and hyperparameters remain development choices, not proof of generalization.

In [27]:
# ============================================================
# STEP 12A: CREATE A REGULARIZED RANDOM FOREST
# ============================================================

# Create a new pipeline using the same preprocessing steps.
# This allows a consistent comparison with our original model.

regularized_rf_model = Pipeline(
    steps=[
        # Apply numerical and categorical preprocessing.
        ("preprocessor", preprocessor),

        # Train Random Forest with restricted tree complexity.
        ("classifier", RandomForestClassifier(
            n_estimators=200,       # Build 200 decision trees.
            max_depth=10,          # Limit each tree to 10 levels.
            min_samples_split=10,  # Require at least 10 samples to split.
            min_samples_leaf=5,    # Require at least 5 samples in each leaf.
            random_state=42,       # Make results reproducible.
            n_jobs=-1              # Use available CPU cores.
        ))
    ]
)

print("Regularized Random Forest pipeline created!")

Regularized Random Forest pipeline created!


In [28]:
# ============================================================
# STEP 12B: TRAIN REGULARIZED RANDOM FOREST
# ============================================================

# Train the new model using the same training observations.
regularized_rf_model.fit(
    X_train_model,
    y_train
)

print("Regularized Random Forest trained successfully!")

Regularized Random Forest trained successfully!


In [29]:
# ============================================================
# STEP 12C: EVALUATE REGULARIZED RANDOM FOREST
# ============================================================

# Generate predictions for the training observations.
regularized_rf_train_pred = regularized_rf_model.predict(
    X_train_model
)

# Generate predictions for the validation observations.
regularized_rf_valid_pred = regularized_rf_model.predict(
    X_valid_model
)

# Calculate training accuracy.
regularized_rf_train_accuracy = accuracy_score(
    y_train,
    regularized_rf_train_pred
)

# Calculate validation accuracy.
regularized_rf_accuracy = accuracy_score(
    y_valid,
    regularized_rf_valid_pred
)

# Calculate the generalization gap.
regularized_rf_gap = (
    regularized_rf_train_accuracy - regularized_rf_accuracy
) * 100

# Display the results.
print("REGULARIZED RANDOM FOREST PERFORMANCE")
print("-" * 50)

print(
    f"Training Accuracy: "
    f"{regularized_rf_train_accuracy * 100:.2f}%"
)

print(
    f"Validation Accuracy: "
    f"{regularized_rf_accuracy * 100:.2f}%"
)

print(
    f"Generalization Gap: "
    f"{regularized_rf_gap:.2f} percentage points"
)

REGULARIZED RANDOM FOREST PERFORMANCE
--------------------------------------------------
Training Accuracy: 84.37%
Validation Accuracy: 79.82%
Generalization Gap: 4.55 percentage points


In [30]:
# ============================================================
# STEP 12D: COMPARE ALL CLASSIFICATION EXPERIMENTS
# ============================================================

# Create a table of training and validation performance.
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Original Random Forest",
        "Regularized Random Forest"
    ],

    "Training_Accuracy": [
        log_train_accuracy,
        rf_train_accuracy,
        regularized_rf_train_accuracy
    ],

    "Validation_Accuracy": [
        baseline_accuracy,
        rf_accuracy,
        regularized_rf_accuracy
    ]
})

# Calculate the difference between training and validation.
model_comparison["Generalization_Gap"] = (
    model_comparison["Training_Accuracy"]
    - model_comparison["Validation_Accuracy"]
)

# Convert accuracy values into percentages.
percentage_columns = [
    "Training_Accuracy",
    "Validation_Accuracy",
    "Generalization_Gap"
]

model_comparison[percentage_columns] = (
    model_comparison[percentage_columns] * 100
)

# Sort models by validation accuracy.
model_comparison = model_comparison.sort_values(
    by="Validation_Accuracy",
    ascending=False
)

# Display the results.
display(model_comparison.round(2))

,Model,Training_Accuracy,Validation_Accuracy,Generalization_Gap
1,Original Random Forest,99.94,80.68,19.26
2,Regularized Random Forest,84.37,79.82,4.55
0,Logistic Regression,79.03,78.84,0.20


## 12. CatBoost classifier

Use CatBoost’s native handling of numerical missing values and categorical predictors (converted to strings, with missing values represented as an explicit category). Unlike Logistic Regression, CatBoost does not require one-hot encoding here.

In [31]:
# ============================================================
# STEP 13A: IMPORT CATBOOST CLASSIFIER
# ============================================================

# Import CatBoost for classification.
from catboost import CatBoostClassifier

print("CatBoost imported successfully!")

CatBoost imported successfully!


In [32]:
# ============================================================
# STEP 13B: PREPARE TRAINING AND VALIDATION DATA FOR CATBOOST
# ============================================================

# Create copies of the original training and validation subsets.
# We do not modify the datasets used by previous models.
X_train_cat = X_train.copy()
X_valid_cat = X_valid.copy()

# Convert categorical variables to strings.
# Missing categories become the explicit label "Missing".
for column in categorical_features:

    X_train_cat[column] = (
        X_train_cat[column]
        .astype("string")
        .fillna("Missing")
        .astype(str)
    )

    X_valid_cat[column] = (
        X_valid_cat[column]
        .astype("string")
        .fillna("Missing")
        .astype(str)
    )

# CatBoost can handle missing numerical values natively.
# Therefore, we leave numerical missing values unchanged.

print("CatBoost training dataset:", X_train_cat.shape)
print("CatBoost validation dataset:", X_valid_cat.shape)

# Check whether categorical variables still contain missing values.
print(
    "Missing categorical values in training:",
    X_train_cat[categorical_features].isna().sum().sum()
)

print(
    "Missing categorical values in validation:",
    X_valid_cat[categorical_features].isna().sum().sum()
)

CatBoost training dataset: (6954, 13)
CatBoost validation dataset: (1739, 13)
Missing categorical values in training: 0
Missing categorical values in validation: 0


In [33]:
# ============================================================
# STEP 13C: CREATE CATBOOST CLASSIFIER
# ============================================================

# Initialize a CatBoost classification model.
catboost_model = CatBoostClassifier(

    iterations=500,       # Maximum number of boosting iterations.

    depth=6,              # Maximum depth of individual trees.

    learning_rate=0.05,   # Controls the contribution of each tree.

    loss_function="Logloss",  # Binary classification objective.

    random_seed=42,       # Make training reproducible.

    verbose=False         # Suppress training iteration messages.
)

print("CatBoost model created successfully!")

CatBoost model created successfully!


In [34]:
# ============================================================
# STEP 13D: TRAIN CATBOOST CLASSIFIER
# ============================================================

# Fit CatBoost using the original training subset.
# The categorical feature names tell CatBoost which columns
# should be handled as categorical predictors.
catboost_model.fit(
    X_train_cat,
    y_train,
    cat_features=categorical_features
)

print("CatBoost trained successfully!")

CatBoost trained successfully!


In [35]:
# ============================================================
# STEP 13E: EVALUATE CATBOOST
# ============================================================

# Generate predictions for training observations.
cat_train_pred = catboost_model.predict(X_train_cat).ravel()

# Generate predictions for validation observations.
cat_valid_pred = catboost_model.predict(X_valid_cat).ravel()

# Calculate training accuracy.
cat_train_accuracy = accuracy_score(
    y_train,
    cat_train_pred
)

# Calculate validation accuracy.
cat_valid_accuracy = accuracy_score(
    y_valid,
    cat_valid_pred
)

# Calculate the generalization gap in percentage points.
cat_gap = (
    cat_train_accuracy - cat_valid_accuracy
) * 100

# Display model performance.
print("CATBOOST PERFORMANCE")
print("-" * 50)

print(
    f"Training Accuracy: {cat_train_accuracy * 100:.2f}%"
)

print(
    f"Validation Accuracy: {cat_valid_accuracy * 100:.2f}%"
)

print(
    f"Generalization Gap: {cat_gap:.2f} percentage points"
)

# Display precision, recall, and F1-score for both classes.
print("\nCATBOOST CLASSIFICATION REPORT")
print("-" * 50)

print(
    classification_report(
        y_valid,
        cat_valid_pred,
        labels=[False, True],
        target_names=["Not Transported", "Transported"],
        digits=4
    )
)

CATBOOST PERFORMANCE
--------------------------------------------------
Training Accuracy: 86.04%
Validation Accuracy: 80.45%
Generalization Gap: 5.59 percentage points

CATBOOST CLASSIFICATION REPORT
--------------------------------------------------
                 precision    recall  f1-score   support

Not Transported     0.8073    0.7961    0.8016       863
    Transported     0.8018    0.8128    0.8073       876

       accuracy                         0.8045      1739
      macro avg     0.8045    0.8044    0.8044      1739
   weighted avg     0.8045    0.8045    0.8045      1739



In [36]:
# ============================================================
# STEP 13F: COMPARE ALL FOUR CLASSIFICATION MODELS
# ============================================================

# Build a comparison table using all four experiments.
all_models_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Original Random Forest",
        "Regularized Random Forest",
        "CatBoost"
    ],

    "Training_Accuracy": [
        log_train_accuracy,
        rf_train_accuracy,
        regularized_rf_train_accuracy,
        cat_train_accuracy
    ],

    "Validation_Accuracy": [
        baseline_accuracy,
        rf_accuracy,
        regularized_rf_accuracy,
        cat_valid_accuracy
    ]
})

# Calculate the generalization gap.
all_models_comparison["Generalization_Gap"] = (
    all_models_comparison["Training_Accuracy"]
    - all_models_comparison["Validation_Accuracy"]
)

# Convert results into percentages.
percentage_columns = [
    "Training_Accuracy",
    "Validation_Accuracy",
    "Generalization_Gap"
]

all_models_comparison[percentage_columns] = (
    all_models_comparison[percentage_columns] * 100
)

# Sort by validation accuracy.
all_models_comparison = all_models_comparison.sort_values(
    by="Validation_Accuracy",
    ascending=False
)

# Display the comparison.
display(all_models_comparison.round(2))

,Model,Training_Accuracy,Validation_Accuracy,Generalization_Gap
1,Original Random Forest,99.94,80.68,19.26
3,CatBoost,86.04,80.45,5.59
2,Regularized Random Forest,84.37,79.82,4.55
0,Logistic Regression,79.03,78.84,0.20


## 13. Five-fold stratified cross-validation

Evaluate the existing scikit-learn pipelines using consistent folds on the development-training subset. The fold-to-fold standard deviation reflects variation across these five partitions, not a confidence interval.

In [37]:
# ============================================================
# STEP 14A: IMPORT CROSS-VALIDATION TOOLS
# ============================================================

# StratifiedKFold creates folds that preserve target proportions.
from sklearn.model_selection import StratifiedKFold

# cross_val_score evaluates a model across multiple folds.
from sklearn.model_selection import cross_val_score

print("Cross-validation tools imported successfully!")

Cross-validation tools imported successfully!


In [38]:
# ============================================================
# STEP 14B: DEFINE FIVE-FOLD STRATIFIED CROSS-VALIDATION
# ============================================================

# Create a reproducible five-fold validation strategy.
cv_strategy = StratifiedKFold(
    n_splits=5,        # Divide data into five folds.
    shuffle=True,      # Shuffle observations before splitting.
    random_state=42    # Ensure reproducible folds.
)

print("Five-fold stratified cross-validation configured!")

Five-fold stratified cross-validation configured!


In [39]:
# ============================================================
# STEP 14C: CROSS-VALIDATE LOGISTIC REGRESSION
# ============================================================

# Evaluate Logistic Regression using five validation folds.
log_cv_scores = cross_val_score(
    logistic_model,
    X_train_model,
    y_train,
    cv=cv_strategy,
    scoring="accuracy",
    n_jobs=-1
)

# Display accuracy for each fold.
print("LOGISTIC REGRESSION CROSS-VALIDATION")

print("Fold accuracies:", log_cv_scores.round(4))

# Calculate the mean accuracy.
print(
    f"Mean accuracy: {log_cv_scores.mean() * 100:.2f}%"
)

# Calculate the standard deviation across folds.
print(
    f"Standard deviation: {log_cv_scores.std() * 100:.2f} percentage points"
)

LOGISTIC REGRESSION CROSS-VALIDATION
Fold accuracies: [0.7879 0.7944 0.7937 0.7822 0.7914]
Mean accuracy: 78.99%
Standard deviation: 0.45 percentage points


In [40]:
# ============================================================
# STEP 14D: CROSS-VALIDATE ORIGINAL RANDOM FOREST
# ============================================================

# Evaluate the original Random Forest across five folds.
rf_cv_scores = cross_val_score(
    random_forest_model,
    X_train_model,
    y_train,
    cv=cv_strategy,
    scoring="accuracy",
    n_jobs=1
)

# Display accuracy for each fold.
print("RANDOM FOREST CROSS-VALIDATION")

print("Fold accuracies:", rf_cv_scores.round(4))

# Calculate the mean accuracy.
print(
    f"Mean accuracy: {rf_cv_scores.mean() * 100:.2f}%"
)

# Calculate standard deviation across folds.
print(
    f"Standard deviation: {rf_cv_scores.std() * 100:.2f} percentage points"
)

RANDOM FOREST CROSS-VALIDATION
Fold accuracies: [0.7958 0.8016 0.8001 0.7872 0.8079]
Mean accuracy: 79.85%
Standard deviation: 0.69 percentage points


In [41]:
# ============================================================
# STEP 14E: CROSS-VALIDATE REGULARIZED RANDOM FOREST
# ============================================================

# Evaluate the regularized Random Forest across five folds.
regularized_rf_cv_scores = cross_val_score(
    regularized_rf_model,
    X_train_model,
    y_train,
    cv=cv_strategy,
    scoring="accuracy",
    n_jobs=1
)

# Display fold accuracies.
print("REGULARIZED RANDOM FOREST CROSS-VALIDATION")

print("Fold accuracies:", regularized_rf_cv_scores.round(4))

# Calculate mean and standard deviation.
print(
    f"Mean accuracy: {regularized_rf_cv_scores.mean() * 100:.2f}%"
)

print(
    f"Standard deviation: "
    f"{regularized_rf_cv_scores.std() * 100:.2f} percentage points"
)

REGULARIZED RANDOM FOREST CROSS-VALIDATION
Fold accuracies: [0.8016 0.8088 0.7958 0.7958 0.8086]
Mean accuracy: 80.21%
Standard deviation: 0.58 percentage points


In [42]:
# ============================================================
# STEP 14F: COMPARE CROSS-VALIDATION RESULTS
# ============================================================

# Create a table containing the mean and standard deviation
# of validation accuracy across the five folds.
cv_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Original Random Forest",
        "Regularized Random Forest"
    ],

    "Mean_CV_Accuracy": [
        log_cv_scores.mean(),
        rf_cv_scores.mean(),
        regularized_rf_cv_scores.mean()
    ],

    "CV_Std": [
        log_cv_scores.std(),
        rf_cv_scores.std(),
        regularized_rf_cv_scores.std()
    ]
})

# Convert proportions into percentages.
cv_comparison["Mean_CV_Accuracy"] *= 100
cv_comparison["CV_Std"] *= 100

# Sort models by their average accuracy.
cv_comparison = cv_comparison.sort_values(
    "Mean_CV_Accuracy",
    ascending=False
)

# Display results rounded to two decimal places.
display(cv_comparison.round(2))

,Model,Mean_CV_Accuracy,CV_Std
2,Regularized Random Forest,80.21,0.58
1,Original Random Forest,79.85,0.69
0,Logistic Regression,78.99,0.45


## 14. CatBoost cross-validation — corrected implementation

The exploratory attempt to use scikit-learn cloning with CatBoost categorical feature settings failed and has been removed from this published version. The successful manual fold loop below freshly creates and trains a CatBoost model for each fold.

In [43]:
# ============================================================
# STEP 15B: CREATE CATBOOST MODEL FOR MANUAL CROSS-VALIDATION
# ============================================================

# Create CatBoost without specifying cat_features
# in the constructor.
catboost_cv_classifier = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    random_seed=42,
    verbose=False
)

print("CatBoost model created successfully!")

CatBoost model created successfully!


In [44]:
# ============================================================
# STEP 15C: MANUAL FIVE-FOLD CATBOOST CROSS-VALIDATION
# ============================================================

# Import the required libraries.
import numpy as np

from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold


# Create a reproducible five-fold validation strategy.
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Create an empty list to store validation accuracy
# from each fold.
cat_cv_scores = []


# Iterate through the five cross-validation folds.
for fold_number, (train_idx, valid_idx) in enumerate(
    cv_strategy.split(X_train, y_train),
    start=1
):

    # --------------------------------------------
    # 1. SELECT TRAINING AND VALIDATION OBSERVATIONS
    # --------------------------------------------

    # Select training observations for the current fold.
    X_fold_train = X_train.iloc[train_idx].copy()

    # Select validation observations for the current fold.
    X_fold_valid = X_train.iloc[valid_idx].copy()

    # Select training target values.
    y_fold_train = y_train.iloc[train_idx]

    # Select validation target values.
    y_fold_valid = y_train.iloc[valid_idx]


    # --------------------------------------------
    # 2. PREPARE CATEGORICAL VARIABLES
    # --------------------------------------------

    # Convert categorical values to strings and
    # replace missing categories with "Missing".
    for column in categorical_features:

        X_fold_train[column] = (
            X_fold_train[column]
            .astype("string")
            .fillna("Missing")
            .astype(str)
        )

        X_fold_valid[column] = (
            X_fold_valid[column]
            .astype("string")
            .fillna("Missing")
            .astype(str)
        )


    # --------------------------------------------
    # 3. CREATE A FRESH CATBOOST MODEL
    # --------------------------------------------

    # A new model must be trained for every fold.
    # This ensures that information learned in one
    # fold does not carry over into another fold.
    fold_model = CatBoostClassifier(
        iterations=500,
        depth=6,
        learning_rate=0.05,
        loss_function="Logloss",
        random_seed=42,
        verbose=False
    )


    # --------------------------------------------
    # 4. TRAIN CATBOOST
    # --------------------------------------------

    # Train using only the current fold's
    # training observations.
    fold_model.fit(
        X_fold_train,
        y_fold_train,
        cat_features=categorical_features
    )


    # --------------------------------------------
    # 5. GENERATE VALIDATION PREDICTIONS
    # --------------------------------------------

    # Predict the transportation outcomes
    # for the current validation fold.
    fold_predictions = fold_model.predict(
        X_fold_valid
    ).ravel()


    # --------------------------------------------
    # 6. CALCULATE VALIDATION ACCURACY
    # --------------------------------------------

    # Compare predicted outcomes with actual outcomes.
    fold_accuracy = accuracy_score(
        y_fold_valid,
        fold_predictions
    )

    # Save this fold's accuracy.
    cat_cv_scores.append(fold_accuracy)

    # Display the result for the current fold.
    print(
        f"Fold {fold_number}: "
        f"{fold_accuracy * 100:.2f}% accuracy"
    )


# --------------------------------------------
# 7. SUMMARIZE CROSS-VALIDATION PERFORMANCE
# --------------------------------------------

# Convert the list of accuracies into a NumPy array.
cat_cv_scores = np.array(cat_cv_scores)

print("\nCATBOOST CROSS-VALIDATION SUMMARY")
print("-" * 50)

# Display accuracy for all five folds.
print("Fold accuracies:", cat_cv_scores.round(4))

# Calculate average cross-validation accuracy.
print(
    f"Mean CV Accuracy: "
    f"{cat_cv_scores.mean() * 100:.2f}%"
)

# Calculate the standard deviation of fold accuracies.
print(
    f"CV Standard Deviation: "
    f"{cat_cv_scores.std() * 100:.2f} percentage points"
)

Fold 1: 80.81% accuracy
Fold 2: 81.09% accuracy
Fold 3: 81.24% accuracy
Fold 4: 81.60% accuracy
Fold 5: 80.86% accuracy

CATBOOST CROSS-VALIDATION SUMMARY
--------------------------------------------------
Fold accuracies: [0.8081 0.8109 0.8124 0.816  0.8086]
Mean CV Accuracy: 81.12%
CV Standard Deviation: 0.28 percentage points


In [45]:
# ============================================================
# STEP 15D: COMPARE ALL FOUR CLASSIFICATION MODELS
# ============================================================

# Create a summary table of cross-validation results.
final_cv_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Original Random Forest",
        "Regularized Random Forest",
        "CatBoost"
    ],

    "Mean_CV_Accuracy": [
        log_cv_scores.mean(),
        rf_cv_scores.mean(),
        regularized_rf_cv_scores.mean(),
        cat_cv_scores.mean()
    ],

    "CV_Std": [
        log_cv_scores.std(),
        rf_cv_scores.std(),
        regularized_rf_cv_scores.std(),
        cat_cv_scores.std()
    ]
})

# Convert results into percentages.
final_cv_comparison["Mean_CV_Accuracy"] *= 100

final_cv_comparison["CV_Std"] *= 100

# Sort models by mean cross-validation accuracy.
final_cv_comparison = final_cv_comparison.sort_values(
    by="Mean_CV_Accuracy",
    ascending=False
)

# Display the results.
display(final_cv_comparison.round(2))

,Model,Mean_CV_Accuracy,CV_Std
3,CatBoost,81.12,0.28
2,Regularized Random Forest,80.21,0.58
1,Original Random Forest,79.85,0.69
0,Logistic Regression,78.99,0.45


## 15. Inspect the original preprocessing output

Inspect the fitted numerical medians, categorical modes and one-hot encoded columns. The original 13 predictors expand to **27 transformed columns** in this fitted pipeline.

In [46]:
# ============================================================
# STEP 27A: INSPECT OUR FITTED PREPROCESSING PIPELINE
# ============================================================

# Retrieve the fitted preprocessing component
# from our trained Logistic Regression pipeline.
fitted_preprocessor = logistic_model.named_steps[
    "preprocessor"
]

# Retrieve the transformed numerical and categorical
# feature names produced by the preprocessing pipeline.
transformed_feature_names = (
    fitted_preprocessor.get_feature_names_out()
)

# Transform the first five training observations.
# This uses the preprocessing rules already learned
# from the training data.
transformed_sample = fitted_preprocessor.transform(
    X_train_model.head(5)
)

# Convert the result to a regular array if the
# preprocessing output is sparse.
if hasattr(transformed_sample, "toarray"):
    transformed_sample = transformed_sample.toarray()

# Create a DataFrame for easier inspection.
transformed_df = pd.DataFrame(
    transformed_sample,
    columns=transformed_feature_names,
    index=X_train_model.head(5).index
)

# Display the transformed observations.
print("Original number of predictors:", X_train_model.shape[1])

print(
    "Number of predictors after preprocessing:",
    transformed_df.shape[1]
)

display(transformed_df)

Original number of predictors: 13
Number of predictors after preprocessing: 27


,num__Age,num__RoomService,num__FoodCourt,num__ShoppingMall,num__Spa,num__VRDeck,num__CabinNumber,cat__HomePlanet_Earth,cat__HomePlanet_Europa,cat__HomePlanet_Mars,...,cat__CabinDeck_A,cat__CabinDeck_B,cat__CabinDeck_C,cat__CabinDeck_D,cat__CabinDeck_E,cat__CabinDeck_F,cat__CabinDeck_G,cat__CabinDeck_T,cat__CabinSide_P,cat__CabinSide_S
3600,-1.996149,-0.33634,-0.279076,-0.305732,-0.267311,-0.264632,0.071038,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1262,-0.811703,-0.33634,-0.279076,-0.305732,-0.267311,-0.264632,-0.777867,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
8612,0.442416,-0.33634,-0.279076,-0.305732,-0.267311,-0.264632,1.758954,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
5075,-0.184643,-0.33634,-0.279076,-0.305732,-0.267311,-0.264632,-0.851082,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
4758,-1.090396,-0.33634,-0.279076,-0.195161,-0.266427,4.423937,0.443052,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


In [47]:
# ============================================================
# STEP 27B: INSPECT LEARNED MISSING-VALUE REPLACEMENTS
# ============================================================

# Retrieve the fitted numerical imputer.
numeric_imputer = (
    fitted_preprocessor
    .named_transformers_["num"]
    .named_steps["imputer"]
)

# Retrieve the fitted categorical imputer.
categorical_imputer = (
    fitted_preprocessor
    .named_transformers_["cat"]
    .named_steps["imputer"]
)

# Show the numerical medians learned from training data.
numerical_imputation_table = pd.DataFrame({
    "Feature": numerical_features,
    "Imputation_Value": numeric_imputer.statistics_
})

print("NUMERICAL IMPUTATION VALUES")
display(numerical_imputation_table)

# Show the most frequent categories used for imputation.
categorical_imputation_table = pd.DataFrame({
    "Feature": categorical_features,
    "Imputation_Value": categorical_imputer.statistics_
})

print("CATEGORICAL IMPUTATION VALUES")
display(categorical_imputation_table)

NUMERICAL IMPUTATION VALUES


,Feature,Imputation_Value
0,Age,27.0
1,RoomService,0.0
2,FoodCourt,0.0
3,ShoppingMall,0.0
4,Spa,0.0
5,VRDeck,0.0
6,CabinNumber,429.0


CATEGORICAL IMPUTATION VALUES


,Feature,Imputation_Value
0,HomePlanet,Earth
1,CryoSleep,False
2,Destination,TRAPPIST-1e
3,VIP,False
4,CabinDeck,F
5,CabinSide,S


## 16. Missingness-aware preprocessing experiment

Compare original median/mode preprocessing to median imputation **with missingness indicators** and an explicit categorical `Missing` label. Both changes occur together, so any accuracy difference cannot be attributed to only one of them.

In [48]:
# ============================================================
# STEP 28A: CREATE AN ALTERNATIVE PREPROCESSING PIPELINE
# ============================================================

# Import the preprocessing tools.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ------------------------------------------------------------
# NUMERICAL PREPROCESSING
# ------------------------------------------------------------

# Replace missing numerical values with the median.
#
# add_indicator=True creates additional binary columns
# identifying features that contained missing values
# during fitting.
numerical_pipeline_v2 = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

# ------------------------------------------------------------
# CATEGORICAL PREPROCESSING
# ------------------------------------------------------------

# Replace missing categorical values with an explicit
# category rather than the most frequent value.
categorical_pipeline_v2 = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Missing"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

# ------------------------------------------------------------
# COMBINE THE PREPROCESSING STEPS
# ------------------------------------------------------------

preprocessor_v2 = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_pipeline_v2,
            numerical_features
        ),
        (
            "cat",
            categorical_pipeline_v2,
            categorical_features
        )
    ]
)

print("Alternative preprocessing pipeline created successfully!")

Alternative preprocessing pipeline created successfully!


In [49]:
# ============================================================
# STEP 28B: CREATE LOGISTIC REGRESSION WITH NEW PREPROCESSING
# ============================================================

from sklearn.linear_model import LogisticRegression

# Create the alternative model pipeline.
logistic_model_v2 = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_v2
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

print("Alternative Logistic Regression model created!")

Alternative Logistic Regression model created!


In [50]:
# ============================================================
# STEP 28C: COMPARE PREPROCESSING STRATEGIES
# ============================================================

import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_score

# Use the same five-fold strategy as our original experiment.
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# ------------------------------------------------------------
# ORIGINAL PREPROCESSING
# ------------------------------------------------------------

# Evaluate our existing Logistic Regression pipeline.
original_cv_scores = cross_val_score(
    logistic_model,
    X_train_model,
    y_train,
    cv=cv_strategy,
    scoring="accuracy",
    n_jobs=1
)

# ------------------------------------------------------------
# ALTERNATIVE PREPROCESSING
# ------------------------------------------------------------

# Evaluate Logistic Regression with missing indicators
# and explicit missing categories.
alternative_cv_scores = cross_val_score(
    logistic_model_v2,
    X_train_model,
    y_train,
    cv=cv_strategy,
    scoring="accuracy",
    n_jobs=1
)

# ------------------------------------------------------------
# COMPARE RESULTS
# ------------------------------------------------------------

comparison_df = pd.DataFrame({
    "Preprocessing": [
        "Median + Mode",
        "Median + Missing Indicators"
    ],
    "Mean_CV_Accuracy": [
        original_cv_scores.mean(),
        alternative_cv_scores.mean()
    ],
    "CV_Std": [
        original_cv_scores.std(),
        alternative_cv_scores.std()
    ]
})

# Convert proportions to percentages.
comparison_df["Mean_CV_Accuracy"] *= 100
comparison_df["CV_Std"] *= 100

# Display results.
display(comparison_df.round(2))

,Preprocessing,Mean_CV_Accuracy,CV_Std
0,Median + Mode,78.99,0.45
1,Median + Missing Indicators,79.16,0.38


## 17. Inspect the alternative feature representation

Inspect the added seven numerical missing indicators and six extra categorical missing-level indicators. The alternative pipeline expands 13 predictors to **40 columns** in this development sample.

In [51]:
# ============================================================
# STEP 29A: INSPECT ALTERNATIVE PREPROCESSING OUTPUT
# ============================================================

# Train the alternative model on the development-training data.
# This is separate from the cross-validation model fits.
logistic_model_v2.fit(
    X_train_model,
    y_train
)

# Retrieve the fitted preprocessing component.
fitted_preprocessor_v2 = logistic_model_v2.named_steps[
    "preprocessor"
]

# Retrieve all transformed feature names.
feature_names_v2 = (
    fitted_preprocessor_v2.get_feature_names_out()
)

# Display the original and transformed feature counts.
print("Original predictors:", X_train_model.shape[1])

print(
    "Original preprocessing feature count:",
    len(transformed_feature_names)
)

print(
    "Alternative preprocessing feature count:",
    len(feature_names_v2)
)

# Display all transformed feature names.
print("\nALTERNATIVE PREPROCESSING FEATURES")
print("-" * 50)

for number, feature in enumerate(feature_names_v2, start=1):
    print(f"{number}. {feature}")

Original predictors: 13
Original preprocessing feature count: 27
Alternative preprocessing feature count: 40

ALTERNATIVE PREPROCESSING FEATURES
--------------------------------------------------
1. num__Age
2. num__RoomService
3. num__FoodCourt
4. num__ShoppingMall
5. num__Spa
6. num__VRDeck
7. num__CabinNumber
8. num__missingindicator_Age
9. num__missingindicator_RoomService
10. num__missingindicator_FoodCourt
11. num__missingindicator_ShoppingMall
12. num__missingindicator_Spa
13. num__missingindicator_VRDeck
14. num__missingindicator_CabinNumber
15. cat__HomePlanet_Earth
16. cat__HomePlanet_Europa
17. cat__HomePlanet_Mars
18. cat__HomePlanet_Missing
19. cat__CryoSleep_False
20. cat__CryoSleep_Missing
21. cat__CryoSleep_True
22. cat__Destination_55 Cancri e
23. cat__Destination_Missing
24. cat__Destination_PSO J318.5-22
25. cat__Destination_TRAPPIST-1e
26. cat__VIP_False
27. cat__VIP_Missing
28. cat__VIP_True
29. cat__CabinDeck_A
30. cat__CabinDeck_B
31. cat__CabinDeck_C
32. cat__Ca

## 18. Compare K-Nearest Neighbors and SVC

Using the same alternative preprocessing and cross-validation folds, compare Logistic Regression with KNN and an RBF Support Vector Classifier. These are untuned learning baselines.

In [52]:
# ============================================================
# STEP 30A: CREATE K-NEAREST NEIGHBORS CLASSIFIER
# ============================================================

# Import KNN classifier.
from sklearn.neighbors import KNeighborsClassifier

# Create a pipeline that combines preprocessing and KNN.
knn_model = Pipeline(
    steps=[
        # Handle missing values, scale numerical features,
        # and one-hot encode categorical variables.
        ("preprocessor", preprocessor_v2),

        # Predict using the five nearest neighbours.
        ("classifier", KNeighborsClassifier(
            n_neighbors=5
        ))
    ]
)

print("KNN classification pipeline created successfully!")

KNN classification pipeline created successfully!


In [53]:
# ============================================================
# STEP 30B: EVALUATE KNN USING CROSS-VALIDATION
# ============================================================

# Evaluate KNN using the same five-fold strategy
# as our earlier classification experiments.
knn_cv_scores = cross_val_score(
    knn_model,
    X_train_model,
    y_train,
    cv=cv_strategy,
    scoring="accuracy",
    n_jobs=1
)

# Display accuracy for each fold.
print("KNN CROSS-VALIDATION RESULTS")
print("-" * 50)

print("Fold accuracies:", knn_cv_scores.round(4))

# Calculate mean cross-validation accuracy.
print(
    f"Mean CV Accuracy: "
    f"{knn_cv_scores.mean() * 100:.2f}%"
)

# Calculate standard deviation across folds.
print(
    f"CV Standard Deviation: "
    f"{knn_cv_scores.std() * 100:.2f} percentage points"
)

KNN CROSS-VALIDATION RESULTS
--------------------------------------------------
Fold accuracies: [0.7721 0.7707 0.7664 0.7728 0.7842]
Mean CV Accuracy: 77.32%
CV Standard Deviation: 0.59 percentage points


In [54]:
# ============================================================
# STEP 30C: CREATE SUPPORT VECTOR CLASSIFIER
# ============================================================

# Import the Support Vector Classifier.
from sklearn.svm import SVC

# Create a preprocessing and classification pipeline.
svc_model = Pipeline(
    steps=[
        # Apply our alternative preprocessing.
        ("preprocessor", preprocessor_v2),

        # Train a nonlinear Support Vector Classifier.
        ("classifier", SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            random_state=42
        ))
    ]
)

print("SVC classification pipeline created successfully!")

SVC classification pipeline created successfully!


In [55]:
# ============================================================
# STEP 30D: FIVE-FOLD CROSS-VALIDATION FOR SVC
# ============================================================

# Evaluate SVC using the same cross-validation strategy.
svc_cv_scores = cross_val_score(
    svc_model,
    X_train_model,
    y_train,
    cv=cv_strategy,
    scoring="accuracy",
    n_jobs=1
)

# Display the cross-validation results.
print("SVC CROSS-VALIDATION RESULTS")
print("-" * 50)

print("Fold accuracies:", svc_cv_scores.round(4))

# Calculate mean accuracy.
print(
    f"Mean CV Accuracy: "
    f"{svc_cv_scores.mean() * 100:.2f}%"
)

# Calculate standard deviation.
print(
    f"CV Standard Deviation: "
    f"{svc_cv_scores.std() * 100:.2f} percentage points"
)

SVC CROSS-VALIDATION RESULTS
--------------------------------------------------
Fold accuracies: [0.7987 0.793  0.7922 0.7922 0.8079]
Mean CV Accuracy: 79.68%
CV Standard Deviation: 0.61 percentage points


In [56]:
# ============================================================
# STEP 30E: COMPARE THREE CLASSIFICATION ALGORITHMS
# ============================================================

# Create a comparison table.
algorithm_comparison = pd.DataFrame({
    "Algorithm": [
        "Logistic Regression",
        "K-Nearest Neighbors",
        "Support Vector Classifier"
    ],

    "Mean_CV_Accuracy": [
        alternative_cv_scores.mean(),
        knn_cv_scores.mean(),
        svc_cv_scores.mean()
    ],

    "CV_Std": [
        alternative_cv_scores.std(),
        knn_cv_scores.std(),
        svc_cv_scores.std()
    ]
})

# Convert accuracy metrics to percentages.
algorithm_comparison["Mean_CV_Accuracy"] *= 100
algorithm_comparison["CV_Std"] *= 100

# Sort by mean cross-validation accuracy.
algorithm_comparison = algorithm_comparison.sort_values(
    by="Mean_CV_Accuracy",
    ascending=False
)

# Display the results.
display(algorithm_comparison.round(2))

,Algorithm,Mean_CV_Accuracy,CV_Std
2,Support Vector Classifier,79.68,0.61
0,Logistic Regression,79.16,0.38
1,K-Nearest Neighbors,77.32,0.59


## 19. Summary of model development

The mean five-fold accuracy scores observed during development were approximately **78.99%** for the original Logistic Regression, **79.85%** for the original Random Forest, **80.21%** for the regularized Random Forest, and **81.12%** for CatBoost. Under the alternative missingness-aware preprocessing, Logistic Regression reached **79.16%**, KNN **77.32%**, and RBF SVC **79.68%**.

**Selected model:** CatBoost (`iterations=500`, `depth=6`, `learning_rate=0.05`). The separate Kaggle submission notebook documents the corresponding full-training fit and public leaderboard result.

**Cautions:** these scores were measured during model development using repeated access to the same development folds. They should not be described as independent post-selection test results. The additional algorithms use initial rather than fully tuned settings, and missingness-aware preprocessing changes both numerical and categorical handling at once.

**Reproducibility:** run the notebook with a fresh kernel after placing `train.csv` and `test.csv` in the project root. Required packages include `pandas`, `numpy`, `scikit-learn`, `catboost`, and `jupyter`.